# MEDICAL IMAGE TYPE CLASSIFIER
### Sirf Run All karo — sab automatic hoga

In [ ]:
# STEP 1: Drive Mount
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# STEP 2: Kaggle Setup
import os, json

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": "mdkaif199", "key": "925011008636896d995996fed323358a"}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!pip install kaggle -q
print('Kaggle ready!')

In [ ]:
# STEP 3: Download Datasets
import os

BASE = '/content/drive/MyDrive/Datasets/image_type_classifier'
for cls in ['brain', 'chest', 'breast', 'skin']:
    os.makedirs(f'{BASE}/{cls}', exist_ok=True)

print('Downloading Brain MRI...')
!kaggle datasets download -d sartajbhuvaji/brain-tumor-classification-mri -p /tmp/brain --unzip -q

print('Downloading Chest X-ray...')
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /tmp/chest --unzip -q

print('Downloading Breast Ultrasound...')
!kaggle datasets download -d aryashah2k/breast-ultrasound-images-dataset -p /tmp/breast --unzip -q

print('Downloading Skin...')
!kaggle datasets download -d kmader/skin-lesion-analysis-toward-melanoma-detection -p /tmp/skin --unzip -q

print('All downloads done!')

In [ ]:
# STEP 4: Copy Images to Drive folders
import shutil, random, glob

def copy_images(src_dir, dst_dir, n=1000):
    imgs = []
    for ext in ['jpg', 'jpeg', 'png']:
        imgs += glob.glob(f'{src_dir}/**/*.{ext}', recursive=True)
        imgs += glob.glob(f'{src_dir}/**/*.{ext.upper()}', recursive=True)
    random.shuffle(imgs)
    imgs = imgs[:n]
    os.makedirs(dst_dir, exist_ok=True)
    for i, src in enumerate(imgs):
        ext = os.path.splitext(src)[-1]
        shutil.copy(src, f'{dst_dir}/{i:05d}{ext}')
    print(f'  {dst_dir}: {len(imgs)} images copied')

copy_images('/tmp/brain',  f'{BASE}/brain',  1000)
copy_images('/tmp/chest',  f'{BASE}/chest',  1000)
copy_images('/tmp/breast', f'{BASE}/breast', 1000)
copy_images('/tmp/skin',   f'{BASE}/skin',   1000)

print('Done!')

In [ ]:
# STEP 5: Build Dataset Structure (train/val split)
import shutil, random, glob, os

DATASET = '/content/med_type_dataset'
CLASSES = ['brain', 'chest', 'breast', 'skin']

for cls in CLASSES:
    imgs = glob.glob(f'{BASE}/{cls}/*')
    random.shuffle(imgs)
    split = int(len(imgs) * 0.8)
    train_imgs = imgs[:split]
    val_imgs   = imgs[split:]

    os.makedirs(f'{DATASET}/train/{cls}', exist_ok=True)
    os.makedirs(f'{DATASET}/val/{cls}',   exist_ok=True)

    for src in train_imgs:
        shutil.copy(src, f'{DATASET}/train/{cls}/{os.path.basename(src)}')
    for src in val_imgs:
        shutil.copy(src, f'{DATASET}/val/{cls}/{os.path.basename(src)}')

    print(f'{cls}: {len(train_imgs)} train, {len(val_imgs)} val')

print('Dataset ready!')

In [ ]:
# STEP 6: Train Model
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Input, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

IMG_SIZE   = 224
BATCH_SIZE = 32

train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True,
)
val_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_data = train_gen.flow_from_directory(
    f'{DATASET}/train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
)
val_data = val_gen.flow_from_directory(
    f'{DATASET}/val',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
)

print('Classes:', train_data.class_indices)

# Build Model
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

inputs  = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = base(inputs, training=True)
x       = GlobalAveragePooling2D()(x)
x       = Dropout(0.4)(x)
x       = Dense(128, activation='relu')(x)
x       = Dropout(0.3)(x)
outputs = Dense(len(CLASSES), activation='softmax')(x)

model = Model(inputs, outputs)
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

# Train
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2),
    ],
)

loss, acc = model.evaluate(val_data)
print(f'\nFinal Accuracy: {acc*100:.2f}%')

In [ ]:
# STEP 7: Save Model to Drive
import json

save_dir = '/content/drive/MyDrive/ml_models'
os.makedirs(save_dir, exist_ok=True)

model.save(f'{save_dir}/image_type_classifier.h5')

with open(f'{save_dir}/image_type_classes.json', 'w') as f:
    json.dump(train_data.class_indices, f)

print('DONE!')
print('Files saved:')
print(f'  {save_dir}/image_type_classifier.h5')
print(f'  {save_dir}/image_type_classes.json')
print('Classes:', train_data.class_indices)